# В данном ноутбуке представлены последовательные шаги по проекту «Влияние на сроки доставки и оценки покупателей маркетплейса Olist».

1. Предобработка данных
2. Формулировка гипотез
3. EDA
4. Апробация
5. Выводы для презентации проектного решения

Над проектом работали:
- Ильин Иван
- Нерода Дарья
- Чащина Алина
- Тимошенко Анастасия

## Предобработка данных

---
### A. Загрузка и предобработка данных

> ◆ **Импорт CSV-файлов с данными Olist**

In [1]:
# импорт необходимых библиотек
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# загрузка датасетов (файлы лежат в папке data/)
DATA_DIR = 'data/'

orders    = pd.read_csv(DATA_DIR + 'olist_orders_dataset.csv')
items     = pd.read_csv(DATA_DIR + 'olist_order_items_dataset.csv')
reviews   = pd.read_csv(DATA_DIR + 'olist_order_reviews_dataset.csv')
payments  = pd.read_csv(DATA_DIR + 'olist_order_payments_dataset.csv')
customers = pd.read_csv(DATA_DIR + 'olist_customers_dataset.csv')
sellers   = pd.read_csv(DATA_DIR + 'olist_sellers_dataset.csv')
products  = pd.read_csv(DATA_DIR + 'olist_products_dataset.csv')
geo       = pd.read_csv(DATA_DIR + 'olist_geolocation_dataset.zip')   # большой файл, хранится в архиве
cat_tr    = pd.read_csv(DATA_DIR + 'product_category_name_translation.csv')

orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [3]:
# размер всех таблиц
tables = {'orders': orders, 'items': items, 'reviews': reviews, 'payments': payments,
          'customers': customers, 'sellers': sellers, 'products': products,
          'geo': geo, 'cat_tr': cat_tr}

pd.DataFrame({'Таблица': tables.keys(),
              'Строк': [t.shape[0] for t in tables.values()],
              'Столбцов': [t.shape[1] for t in tables.values()]})

,Таблица,Строк,Столбцов
0,orders,99441,8
1,items,112650,7
2,reviews,99224,7
3,payments,103886,5
4,customers,99441,5
5,sellers,3095,4
6,products,32951,9
7,geo,1000163,5
8,cat_tr,71,2


**Вывод:** загружено 9 таблиц. Главная таблица — заказы (orders), остальные связаны с ней через номер заказа, покупателя, товара или продавца.

> ◆ **Проверка пропущенных значений и приведение переменных к корректным типам**

In [4]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB


In [5]:
qa = pd.DataFrame({
    'Таблица': tables.keys(),
    'Пропусков всего': [t.isna().sum().sum() for t in tables.values()],
    'Дубликаты строк': [t.duplicated().sum() for t in tables.values()],
})
display(qa)

,Таблица,Пропусков всего,Дубликаты строк
0,orders,4908,0
1,items,0,0
2,reviews,145903,0
3,payments,0,0
4,customers,0,0
5,sellers,0,0
6,products,2448,0
7,geo,0,261831
8,cat_tr,0,0


In [6]:
# в каких столбцах заказов и товаров есть пропуски
print('Заказы:')
print(orders.isna().sum()[orders.isna().sum() > 0])
print()
print('Товары:')
print(products.isna().sum()[products.isna().sum() > 0])

Заказы:
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

Товары:
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64


**Вывод:** пропуски есть в датах заказов (подтверждение, передача перевозчику, доставка) и в описании товаров — у части товаров нет категории. В отзывах пусто в заголовке и тексте — покупатели часто ставят только оценку. Полные дубликаты строк есть только в таблице геолокации.

> ◆ **Обработка временных меток (даты покупки, доставки и обещанной доставки)**

In [7]:
# даты записаны как текст - переводим в формат даты
date_cols = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date',
             'order_delivered_customer_date', 'order_estimated_delivery_date']
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

orders[date_cols].dtypes

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]

In [8]:
# пропуски дат доставки - у каких заказов?
orders[orders['order_delivered_customer_date'].isna()]['order_status'].value_counts()

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2

**Вывод:** даты переведены в формат даты. Пропуски даты доставки в основном у заказов, которые ещё не доставлены или отменены, — для анализа доставки будем брать доставленные заказы.

> ◆ **Несколько товаров, оплат и отзывов в одном заказе**

In [9]:
print('Заказов с несколькими товарами:', (items.groupby('order_id').size() > 1).sum())
print('Заказов с несколькими оплатами:', (payments.groupby('order_id').size() > 1).sum())
print('Заказов с несколькими отзывами:', (reviews.groupby('order_id').size() > 1).sum())

Заказов с несколькими товарами: 9803
Заказов с несколькими оплатами: 2961
Заказов с несколькими отзывами: 547


**Вывод:** при объединении таблиц строки размножатся, поэтому товары, оплаты и отзывы нужно сначала свести к одной строке на заказ.

> ◆ **Странные даты и период данных**

In [10]:
delivered = orders[orders['order_status'] == 'delivered']
print('Передача перевозчику раньше покупки:',
      (delivered['order_delivered_carrier_date'] < delivered['order_purchase_timestamp']).sum())
print('Доставка раньше передачи перевозчику:',
      (delivered['order_delivered_customer_date'] < delivered['order_delivered_carrier_date']).sum())

Передача перевозчику раньше покупки: 165
Доставка раньше передачи перевозчику: 23


In [11]:
# количество заказов по месяцам
orders['order_purchase_timestamp'].dt.to_period('M').value_counts().sort_index()

order_purchase_timestamp
2016-09       4
2016-10     324
2016-12       1
2017-01     800
2017-02    1780
2017-03    2682
2017-04    2404
2017-05    3700
2017-06    3245
2017-07    4026
2017-08    4331
2017-09    4285
2017-10    4631
2017-11    7544
2017-12    5673
2018-01    7269
2018-02    6728
2018-03    7211
2018-04    6939
2018-05    6873
2018-06    6167
2018-07    6292
2018-08    6512
2018-09      16
2018-10       4
Freq: M

**Вывод:** есть небольшое число заказов со странными датами — нужно решить, удалять ли их. В 2016 году и в сентябре–октябре 2018 заказов очень мало, эти месяцы, скорее всего, уберём из анализа.

> ◆ **Таблица геолокации**

In [12]:
print('Дубликатов строк:', geo.duplicated().sum())
in_brazil = geo['geolocation_lat'].between(-34, 6) & geo['geolocation_lng'].between(-74, -34)
print('Координаты за пределами Бразилии:', (~in_brazil).sum())

Дубликатов строк: 261831
Координаты за пределами Бразилии: 42


**Вывод:** в таблице геолокации много повторов и есть ошибочные координаты. Нужна ли она для анализа — уточним у заказчика.

> ◆ **Вопросы, которые нужно уточнить у заказчика**

- Что считать опозданием — доставку позже обещанной даты? Сравнивать по дню или по часам?
- Удалять ли заказы со странными датами?
- Нужна ли таблица геолокации (расстояние между продавцом и покупателем)?

---
### B. Описательная статистика

> ◆ **Распределение заказов по месяцам, штатам и статусам**

> ◆ **Расчёт основных показателей (среднее, медиана) для срока доставки, стоимости и оценок**

### C. Визуализация ключевых переменных

> ◆ **Гистограмма срока доставки**

> ◆ **Доля опозданий по месяцам и штатам**

> ◆ **Оценки покупателей у опоздавших и доставленных вовремя заказов**

### D. Анализ взаимосвязей

> ◆ **Корреляционный анализ между сроком доставки, стоимостью и оценкой**

> ◆ **Визуализация корреляционной матрицы**

### E. Проверка статистических гипотез

> ◆ **Сравнение оценок опоздавших и доставленных вовремя заказов**

> ◆ **Сравнение доли опозданий по группам (штат, месяц, продавец)**

### F. Выводы и рекомендации

- Какие факторы чаще всего приводят к опозданиям?
- Насколько опоздание влияет на оценку покупателя?
- Какие регионы и месяцы требуют особого внимания?
- Рекомендации по сокращению задержек доставки.